In [ ]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

In [ ]:
import os
import site
import polars as pl

# -----------------------------
# 1) Read training data
# -----------------------------
train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(train.head())

# -----------------------------
# 2) Add CUTLASS package path
# -----------------------------
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

# -----------------------------
# 3) Imports
# -----------------------------
import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# -----------------------------
# 4) Configuration
# -----------------------------
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Competition max is 32

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LORA_RANK:", LORA_RANK)

# -----------------------------
# 5) Load base model
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

print("Model loaded successfully.")

# -----------------------------
# 6) Load tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
print("Tokenizer loaded successfully.")

# -----------------------------
# 7) Initialize LoRA
# -----------------------------
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# -----------------------------
# 8) No training yet
# -----------------------------
# model.train()
# YOUR TRAINING CODE HERE

# -----------------------------
# 9) Save adapter artifacts
# -----------------------------
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

# Optional: save tokenizer files too
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:")
print(sorted(os.listdir(OUTPUT_DIR)))

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')